***Step 1 — Set Up Data Lake & Build Bronze Layer***

In [15]:
import os
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.neighbors import NearestNeighbors

# 1. Create Medallion Data Lake folders
for folder in ['bronze', 'silver', 'gold']:
    os.makedirs(folder, exist_ok=True)

# 2. Verify local files are present
required_files = ["BX-Users.csv", "BX-Books.csv", "BX-Book-Ratings.csv"]
assert all(os.path.exists(f) for f in required_files), (
    f"Missing one or more required CSV files in the current directory. "
    f"Please make sure {required_files} are uploaded here."
)

# 3. Load raw files exactly as they are (semicolon separators, latin-1 encoding)
users_raw = pd.read_csv("BX-Users.csv", sep=";", encoding="latin-1", low_memory=False)
books_raw = pd.read_csv("BX-Books.csv", sep=";", encoding="latin-1", on_bad_lines="skip", low_memory=False)
ratings_raw = pd.read_csv("BX-Book-Ratings.csv", sep=";", encoding="latin-1", low_memory=False)

# 4. Add ingestion_timestamp column (zero cleaning in Bronze)
ingest_ts = pd.Timestamp.utcnow()
for df in [users_raw, books_raw, ratings_raw]:
    df['ingestion_timestamp'] = ingest_ts

# Ensure mixed-type Year-Of-Publication serializes cleanly to Parquet without modifying raw values
books_raw['Year-Of-Publication'] = books_raw['Year-Of-Publication'].astype(str)

# 5. Save untouched Bronze tables as Parquet
users_raw.to_parquet("bronze/users.parquet", index=False)
books_raw.to_parquet("bronze/books.parquet", index=False)
ratings_raw.to_parquet("bronze/ratings.parquet", index=False)

print("Bronze layer created successfully from local files:", os.listdir("bronze"))

Bronze layer created successfully from local files: ['ratings.parquet', 'books.parquet', 'users.parquet']


***Step 2 — Explore the Bronze Data (EDA)***

In [16]:
# Read Bronze tables back
b_users = pd.read_parquet("bronze/users.parquet")
b_books = pd.read_parquet("bronze/books.parquet")
b_ratings = pd.read_parquet("bronze/ratings.parquet")

# 1. Shape, data types, and missing-value count
for name, df in [("Users", b_users), ("Books", b_books), ("Ratings", b_ratings)]:
    print(f"=== {name} ===")
    print("Shape:", df.shape)
    print("Dtypes:\n", df.dtypes)
    print("Missing Values:\n", df.isnull().sum(), "\n")

# 2. Distribution of ratings (noting 0s)
rating_dist = b_ratings['Book-Rating'].value_counts().sort_index()
print("=== Rating Distribution ===")
print(rating_dist)
print(f"Rating = 0 count: {rating_dist[0]:,} ({rating_dist[0] / len(b_ratings) * 100:.2f}%)\n")

# 3. Distribution of user ages and publication years
print("=== User Age Distribution ===")
print(b_users['Age'].describe())
print("Impossible ages (< 5 or > 100):", b_users[(b_users['Age'] < 5) | (b_users['Age'] > 100)].shape[0])

pub_years = pd.to_numeric(b_books['Year-Of-Publication'], errors='coerce')
print("\n=== Publication Year Distribution ===")
print(pub_years.describe())
print("Impossible years (<= 0 or > 2026 or invalid):", pub_years[(pub_years <= 0) | (pub_years > 2026) | (pub_years.isna())].shape[0])

# 4. Number of ratings per user and per book
print("\n=== Ratings per User ===")
print(b_ratings.groupby('User-ID')['Book-Rating'].count().describe(percentiles=[0.5, 0.75, 0.9, 0.95]))

print("\n=== Ratings per Book ===")
print(b_ratings.groupby('ISBN')['Book-Rating'].count().describe(percentiles=[0.5, 0.75, 0.9, 0.95]))

# 5. Matrix sparsity (percentage of possible user-book pairs with a rating)
n_users = b_ratings['User-ID'].nunique()
n_books = b_ratings['ISBN'].nunique()
populated_pct = (len(b_ratings) / (n_users * n_books)) * 100
print(f"\n=== Matrix Sparsity ===")
print(f"Unique Users: {n_users:,} | Unique Books: {n_books:,}")
print(f"Percentage of possible pairs with a rating: {populated_pct:.6f}% (Empty/Sparse: {100 - populated_pct:.6f}%)")

=== Users ===
Shape: (278858, 4)
Dtypes:
 User-ID                              int64
Location                            object
Age                                float64
ingestion_timestamp    datetime64[us, UTC]
dtype: object
Missing Values:
 User-ID                     0
Location                    0
Age                    110762
ingestion_timestamp         0
dtype: int64 

=== Books ===
Shape: (271360, 9)
Dtypes:
 ISBN                                object
Book-Title                          object
Book-Author                         object
Year-Of-Publication                 object
Publisher                           object
Image-URL-S                         object
Image-URL-M                         object
Image-URL-L                         object
ingestion_timestamp    datetime64[us, UTC]
dtype: object
Missing Values:
 ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-

***Step 2 — EDA Findings & Silver Layer Plan***

### Step 2 — EDA Observations & Silver Layer Plan

* **Shape, data types and missing-value count:** Users has `278,858` rows (`Age` missing in `110,762` rows), Books has `271,360` rows (`Year-Of-Publication` typed as string, `2` missing authors, `2` missing publishers), and Ratings has `1,149,780` rows with 0 nulls; in Silver, we will enforce proper numeric/string types, impute missing ages, and drop unused image URL columns.
* **Distribution of ratings:** `716,109` ratings (`62.28%`) are `0` (implicit interactions) while `433,671` (`37.72%`) are explicit ratings from `1` to `10` (peaking at `8`); in Silver, we will drop all `0` ratings and keep only explicit ratings (`1–10`).
* **Distribution of user ages and publication years:** User ages contain `1,248` impossible values (`< 5` or `> 100`, up to `244`), and publication years contain `4,632` invalid values (`0` or future years up to `2050`); in Silver, we will replace out-of-range ages with null and impute the median (`32.0`), and set invalid publication years to null.
* **Number of ratings per user and per book:** Both are heavily long-tailed with a median of `1` rating per user (`90th` percentile = `12`) and `1` rating per book (`95th` percentile = `10`); in Silver, we will filter to users and books with at least `10` explicit ratings to remove cold-start noise.
* **Matrix sparsity:** Only `0.003207%` of all possible user-book pairs have a rating (`99.996793%` sparse); in Silver, removing `0` ratings and enforcing the minimum rating threshold will make the matrix dense enough for meaningful cosine similarity in Gold.

***Step 3 — Clean the Data & Build Silver Layer***

In [17]:
s_users = pd.read_parquet("bronze/users.parquet").drop(columns=['ingestion_timestamp'])
s_books = pd.read_parquet("bronze/books.parquet").drop(columns=['ingestion_timestamp'])
s_ratings = pd.read_parquet("bronze/ratings.parquet").drop(columns=['ingestion_timestamp'])

# 1. Remove duplicate rows
s_users = s_users.drop_duplicates().drop_duplicates(subset=['User-ID'])
s_books = s_books.drop_duplicates().drop_duplicates(subset=['ISBN'])
s_ratings = s_ratings.drop_duplicates()

# 2. Remove ratings of 0 and keep ratings from 1 to 10
s_ratings = s_ratings[s_ratings['Book-Rating'].between(1, 10)].copy()

# 3. Fix user ages: set values < 5 or > 100 to null, then fill nulls with median age
s_users['Age'] = pd.to_numeric(s_users['Age'], errors='coerce')
s_users.loc[(s_users['Age'] < 5) | (s_users['Age'] > 100), 'Age'] = np.nan
median_age = s_users['Age'].median()
s_users['Age'] = s_users['Age'].fillna(median_age)

# 4. Split Location text into city, state, and country columns
loc_split = s_users['Location'].fillna('').str.split(',', n=2, expand=True)
s_users['city'] = loc_split[0].fillna('').str.strip().str.title()
s_users['state'] = loc_split[1].fillna('').str.strip().str.title()
s_users['country'] = loc_split[2].fillna('').str.strip().str.title()
s_users = s_users.drop(columns=['Location'])

# 5. Fix publication years: set 0 or future years (> 2026) to null
s_books['Year-Of-Publication'] = pd.to_numeric(s_books['Year-Of-Publication'], errors='coerce')
s_books.loc[(s_books['Year-Of-Publication'] <= 0) | (s_books['Year-Of-Publication'] > 2026), 'Year-Of-Publication'] = np.nan

# 6. Clean text columns, standardize case, and drop image-URL columns
s_books = s_books.drop(columns=['Image-URL-S', 'Image-URL-M', 'Image-URL-L'])
s_books['ISBN'] = s_books['ISBN'].astype(str).str.strip().str.upper()
s_books['Book-Title'] = s_books['Book-Title'].astype(str).str.strip()
s_books['Book-Author'] = s_books['Book-Author'].fillna('Unknown').astype(str).str.strip().str.title()
s_books['Publisher'] = s_books['Publisher'].fillna('Unknown').astype(str).str.strip()

s_ratings['ISBN'] = s_ratings['ISBN'].astype(str).str.strip().str.upper()
s_ratings = s_ratings.groupby(['User-ID', 'ISBN'], as_index=False)['Book-Rating'].mean()

# 7. Keep only ratings whose book and user exist in books and users tables
s_ratings = s_ratings[
    s_ratings['User-ID'].isin(s_users['User-ID']) &
    s_ratings['ISBN'].isin(s_books['ISBN'])
].copy()

# 8. Keep only users and books with at least MIN_RATINGS = 10 explicit ratings
MIN_RATINGS = 10
while True:
    prev_rows = len(s_ratings)
    u_counts = s_ratings['User-ID'].value_counts()
    s_ratings = s_ratings[s_ratings['User-ID'].isin(u_counts[u_counts >= MIN_RATINGS].index)]

    b_counts = s_ratings['ISBN'].value_counts()
    s_ratings = s_ratings[s_ratings['ISBN'].isin(b_counts[b_counts >= MIN_RATINGS].index)]

    if len(s_ratings) == prev_rows:
        break

s_users = s_users[s_users['User-ID'].isin(s_ratings['User-ID'])].copy()
s_books = s_books[s_books['ISBN'].isin(s_ratings['ISBN'])].copy()

# Save Silver tables as Parquet
s_users.to_parquet("silver/users.parquet", index=False)
s_books.to_parquet("silver/books.parquet", index=False)
s_ratings.to_parquet("silver/ratings.parquet", index=False)

print(f"Median age imputed: {median_age}")
print(f"Silver layer saved -> Users: {len(s_users):,}, Books: {len(s_books):,}, Ratings: {len(s_ratings):,}")

Median age imputed: 32.0
Silver layer saved -> Users: 1,820, Books: 2,087, Ratings: 41,469


***Step 3 — Minimum Rating Threshold Justification***

### Step 3 — Minimum Rating Threshold Justification (`MIN_RATINGS = 10`)

In Step 2 EDA, over 90% of users and 95% of books had fewer than 10 ratings (median = `1`), resulting in `99.9968%` matrix sparsity where item-to-item cosine similarity suffers from single-user noise. Enforcing a threshold of **10 explicit ratings per user and per book** retains `1,820` active readers, `2,087` well-reviewed books, and `41,469` reliable ratings, raising matrix density from `0.0032%` to `1.09%` for accurate collaborative filtering.

***Step 4 — Build the Gold Layer***

In [18]:
g_users = pd.read_parquet("silver/users.parquet")
g_books = pd.read_parquet("silver/books.parquet")
g_ratings = pd.read_parquet("silver/ratings.parquet")

# Map User-ID and ISBN to contiguous matrix indices
unique_users = np.sort(g_ratings['User-ID'].unique())
unique_books = np.sort(g_ratings['ISBN'].unique())

user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
book_to_idx = {isbn: i for i, isbn in enumerate(unique_books)}

g_ratings['user_idx'] = g_ratings['User-ID'].map(user_to_idx)
g_ratings['book_idx'] = g_ratings['ISBN'].map(book_to_idx)

# 1. Sparse user-book rating matrix
user_book_sparse = csr_matrix(
    (g_ratings['Book-Rating'].values, (g_ratings['user_idx'].values, g_ratings['book_idx'].values)),
    shape=(len(unique_users), len(unique_books))
)
save_npz("gold/user_book_rating_matrix.npz", user_book_sparse)
pd.DataFrame({'user_idx': range(len(unique_users)), 'User-ID': unique_users}).to_parquet("gold/user_map.parquet", index=False)

# 2. Book lookup table (book_idx, ISBN, Book-Title, Book-Author)
book_lookup = pd.DataFrame({'book_idx': range(len(unique_books)), 'ISBN': unique_books}).merge(
    g_books[['ISBN', 'Book-Title', 'Book-Author']], on='ISBN', how='left'
)
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)

# 3. User-level and book-level aggregates (average rating and number of ratings)
user_agg = g_ratings.groupby('User-ID', as_index=False).agg(
    num_ratings=('Book-Rating', 'count'),
    avg_rating=('Book-Rating', 'mean')
)
book_agg = g_ratings.groupby('ISBN', as_index=False).agg(
    num_ratings=('Book-Rating', 'count'),
    avg_rating=('Book-Rating', 'mean')
).merge(g_books[['ISBN', 'Book-Title', 'Book-Author']], on='ISBN', how='left')

user_agg.to_parquet("gold/user_aggregates.parquet", index=False)
book_agg.to_parquet("gold/book_aggregates.parquet", index=False)

print("Gold layer saved:", os.listdir("gold"))
print("Sparse matrix shape (users x books):", user_book_sparse.shape)

Gold layer saved: ['user_aggregates.parquet', 'book_lookup.parquet', 'user_book_rating_matrix.npz', 'book_aggregates.parquet', 'user_map.parquet']
Sparse matrix shape (users x books): (1820, 2030)


***Step 5 — Build the Item-Based Collaborative Filtering Recommender***

In [19]:
# 1. Load Gold rating matrix and arrange so each row is a book and each column is a user
user_book_mat = load_npz("gold/user_book_rating_matrix.npz")
book_user_mat = user_book_mat.T.tocsr()  # Shape: (2087 books, 1820 users)

book_lookup = pd.read_parquet("gold/book_lookup.parquet")
user_map = pd.read_parquet("gold/user_map.parquet")

uid_to_idx = dict(zip(user_map['User-ID'], user_map['user_idx']))
idx_to_title = dict(zip(book_lookup['book_idx'], book_lookup['Book-Title']))
idx_to_author = dict(zip(book_lookup['book_idx'], book_lookup['Book-Author']))

# 2. Compute cosine similarity between books using scikit-learn NearestNeighbors
knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=25)
knn_model.fit(book_user_mat)

# 3. Function taking a book title and returning the 10 most similar books
def similar_books(title, n=10):
    matches = book_lookup[book_lookup['Book-Title'].str.lower() == title.strip().lower()]
    if matches.empty:
        matches = book_lookup[book_lookup['Book-Title'].str.lower().str.contains(title.strip().lower(), regex=False)]
    if matches.empty:
        raise ValueError(f"Book title '{title}' not found in Gold lookup table.")

    book_idx = matches.iloc[0]['book_idx']
    distances, indices = knn_model.kneighbors(book_user_mat[book_idx], n_neighbors=n + 1)

    similar = []
    for dist, idx in zip(distances.flatten(), indices.flatten()):
        if idx == book_idx:
            continue
        similar.append({
            'Book-Title': idx_to_title[idx],
            'Book-Author': idx_to_author[idx],
            'Cosine-Similarity': round(1.0 - dist, 4)
        })
        if len(similar) == n:
            break
    return pd.DataFrame(similar)

# 4. recommend(user_id, n=10)
def recommend(user_id, n=10):
    if user_id not in uid_to_idx:
        raise ValueError(f"User-ID {user_id} not found in Gold dataset.")

    u_idx = uid_to_idx[user_id]
    user_row = user_book_mat[u_idx]
    already_read = set(user_row.indices)

    # Select books the user rated highly (>= 7 or >= user's mean rating)
    user_ratings = dict(zip(user_row.indices, user_row.data))
    high_threshold = max(7.0, np.mean(list(user_ratings.values())))
    highly_rated = {b_idx: rating for b_idx, rating in user_ratings.items() if rating >= high_threshold}
    if not highly_rated:
        highly_rated = user_ratings

    # Collect similar books, weight similarity by user's rating, and exclude already-read books
    scores = {}
    for b_idx, rating in highly_rated.items():
        distances, indices = knn_model.kneighbors(book_user_mat[b_idx], n_neighbors=20)
        for dist, neighbor_idx in zip(distances.flatten(), indices.flatten()):
            if neighbor_idx in already_read:
                continue
            sim = 1.0 - dist
            if sim > 0:
                scores[neighbor_idx] = scores.get(neighbor_idx, 0.0) + (sim * rating)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]
    return [idx_to_title[idx] for idx, _ in ranked]

# Test item-to-item similarity function
print("10 Most Similar Books to 'The Lovely Bones: A Novel':")
display(similar_books("The Lovely Bones: A Novel", n=10))

10 Most Similar Books to 'The Lovely Bones: A Novel':


,Book-Title,Book-Author,Cosine-Similarity
0,Where the Heart Is (Oprah's Book Club (Paperba...,Billie Letts,0.2192
1,Lucky : A Memoir,Alice Sebold,0.2180
2,The Da Vinci Code,Dan Brown,0.1783
3,The Pilot's Wife : A Novel,Anita Shreve,0.1763
4,Good in Bed,Jennifer Weiner,0.1758
5,Nights in Rodanthe,Nicholas Sparks,0.1702
6,The Pact: A Love Story,Jodi Picoult,0.1682
7,The Lake House,James Patterson,0.1662
8,Lucky,Alice Sebold,0.1651
9,White Oleander : A Novel (Oprah's Book Club),Janet Fitch,0.1542


***Step 6 — Demo and Sanity Check***

In [20]:
demo_users = [52521, 277523, 63625]

for uid in demo_users:
    print("=" * 85)
    print(f"DEMO FOR USER-ID: {uid}")

    u_idx = uid_to_idx[uid]
    user_row = user_book_mat[u_idx]
    history_df = pd.DataFrame({
        'Book-Title': [idx_to_title[i] for i in user_row.indices],
        'Book-Author': [idx_to_author[i] for i in user_row.indices],
        'Rating': user_row.data
    }).sort_values(by='Rating', ascending=False)

    print("\nBooks User Rated Highly:")
    for _, row in history_df.head(8).iterrows():
        print(f"  * [{row['Rating']:.0f}/10] {row['Book-Title']} — {row['Book-Author']}")

    recs = recommend(uid, n=10)
    print("\nTop 10 Recommended Books:")
    for rank, title in enumerate(recs, 1):
        author = book_lookup.loc[book_lookup['Book-Title'] == title, 'Book-Author'].iloc[0]
        print(f"  {rank:2d}. {title} — {author}")
    print()

DEMO FOR USER-ID: 52521

Books User Rated Highly:
  * [10/10] The Concrete Blonde (A Harry Bosch Novel) — Michael Connelly
  * [10/10] Macgregor Brides (Macgregors) — Nora Roberts
  * [10/10] Stanislaski Brothers (Silhouette Promo) — Nora Roberts
  * [10/10] Stanislaski Sisters — Nora Roberts
  * [10/10] Cruel &amp; Unusual (Kay Scarpetta Mysteries (Paperback)) — Patricia D. Cornwell
  * [10/10] Until Forever — Johanna Lindsey
  * [10/10] Night Prey — John Sandford
  * [10/10] The Body Farm — Patricia Daniels Cornwell

Top 10 Recommended Books:
   1. Unnatural Exposure — Patricia Daniels Cornwell
   2. All That Remains (Kay Scarpetta Mysteries (Paperback)) — Patricia D. Cornwell
   3. Considering Kate (The Stanislaskis) (Silhouette Special Edition) — Nora Roberts
   4. Born in Fire — Nora Roberts
   5. The Last Precinct — Patricia Daniels Cornwell
   6. The Ritual Bath (Peter Decker &amp; Rina Lazarus Novels (Paperback)) — Faye Kellerman
   7. Born in Ice — Nora Roberts
   8. Black Not